### Agents - Analysis Report Creation

In [1]:
import os
import openai

openai.api_key=os.getenv("OPENAI_API_KEY2")

In [2]:
import requests
from newspaper import Article # https://github.com/codelucas/newspaper
import time

headers = {
    'User-Agent': '''Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36'''
}

article_urls = [
    """https://www.artificialintelligence-news.com/2023/05/23/meta-open-source-speech-ai-models-support-over-1100-languages/""",
    """https://www.artificialintelligence-news.com/2023/05/18/beijing-launches-campaign-against-ai-generated-misinformation/"""
    """https://www.artificialintelligence-news.com/2023/05/16/openai-ceo-ai-regulation-is-essential/""",
    """https://www.artificialintelligence-news.com/2023/05/15/jay-migliaccio-ibm-watson-on-leveraging-ai-to-improve-productivity/""",
    """https://www.artificialintelligence-news.com/2023/05/15/iurii-milovanov-softserve-how-ai-ml-is-helping-boost-innovation-and-personalisation/""",
    """https://www.artificialintelligence-news.com/2023/05/11/ai-and-big-data-expo-north-america-begins-in-less-than-one-week/""",
    """https://www.artificialintelligence-news.com/2023/05/11/eu-committees-green-light-ai-act/""",
    """https://www.artificialintelligence-news.com/2023/05/09/wozniak-warns-ai-will-power-next-gen-scams/""",
    """https://www.artificialintelligence-news.com/2023/05/09/infocepts-ceo-shashank-garg-on-the-da-market-shifts-and-impact-of-ai-on-data-analytics/""",
    """https://www.artificialintelligence-news.com/2023/05/02/ai-godfather-warns-dangers-and-quits-google/""",
    """https://www.artificialintelligence-news.com/2023/04/28/palantir-demos-how-ai-can-used-military/""",
    """https://www.artificialintelligence-news.com/2023/04/26/ftc-chairwoman-no-ai-exemption-to-existing-laws/""",
    """https://www.artificialintelligence-news.com/2023/04/24/bill-gates-ai-teaching-kids-literacy-within-18-months/""",
    """https://www.artificialintelligence-news.com/2023/04/21/google-creates-new-ai-division-to-challenge-openai/"""
]

session = requests.Session()
pages_content = [] # where we save the scraped articles

for url in article_urls:
    try:
        time.sleep(2) # sleep two seconds for gentle scraping
        response = session.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            article = Article(url)
            article.download() # download HTML of webpage
            article.parse() # parse HTML to extract the article text
            pages_content.append({ "url": url, "text": article.text })
        else:
            print(f"Failed to fetch article at {url}")
    except Exception as e:
        print(f"Error occurred while fetching article at {url}: {e}")


In [20]:
pages_text=[]
for i in pages_content:
    pages_text.append(i['text'])

In [24]:
from langchain_community.vectorstores.faiss import FAISS
from langchain_openai.embeddings.base import OpenAIEmbeddings
import os
os.environ["ALLOW_DANGEROUS_DESERIALIZATION"] = "true"
# Assuming `texts` is a list of chunks
texts_docs = pages_text

# Generate embeddings
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')
faiss_index_agent_text = FAISS.from_texts(texts_docs, embeddings)

# Save FAISS index locally
faiss_index_agent_text.save_local("faiss_index_agent_text")

In [26]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

all_texts = []
for d in pages_content:
    chunks = text_splitter.split_text(d["text"])
    for chunk in chunks:
        all_texts.append(chunk)

# Load FAISS index locally
faiss_index_agent_text = FAISS.load_local("faiss_index_agent_text", embeddings, allow_dangerous_deserialization=True)
faiss_index_agent_text.add_texts(all_texts)

['16427b99-5ba6-49e9-81f7-c238a82ab1e7',
 'f541f9b1-1de7-4d87-a086-23fd235dedb1',
 'd439d828-9cd5-4d5e-a7e4-bae76f10a5e8',
 '2249c140-efa4-4876-8e4e-e6a661b3e0e6',
 'b58c3af3-0127-400b-83ea-f9c082254518',
 'fe8fb7a1-67ab-4e86-a189-4bcc2850fef0',
 'c4efd7be-6242-4533-8e9e-fd3a43924b65',
 '463af046-cf04-4a44-ae18-b2b70ecd7df8',
 'f3ada08d-bef7-4776-8d8b-56fb3f3f9d42',
 '7917e591-1097-4eb8-93ca-c577c734da1c',
 '7b81d660-ff62-4aab-837e-7321762ade50',
 '998cadb9-96f5-4a1d-9895-1699c59272bf',
 '8c7177c8-e146-4e77-a607-8a22d9f58f27',
 '9037e3fe-63c6-486a-be5b-c0681839b8af',
 '023b6329-3b0f-48bd-a8b3-15baaae04b23',
 '9a8b77f4-1784-4280-b3fe-2c83416b79ae',
 '90004308-3ee8-437e-8447-298b9cea62ee',
 '670dd2ff-ae4f-41cf-8b5e-24f27012296d',
 '96a5d96c-268c-42e0-b812-a33e0f583b14',
 '0c5f14a9-e399-445f-81d7-4fa493d28216',
 '1ea083e1-19e6-4b89-9a1b-99e0ec71e1ea',
 '938220c3-905c-4a4c-8cb6-c8577be5d3a4',
 '3ea7ee41-ad68-47ac-b16a-7252b0ae4f7b',
 'fc95ad9c-8bd5-4a4e-8717-7d47e6620700',
 'd1b02e28-e443-

In [30]:
retriever = faiss_index_agent_text.as_retriever()
retriever.search_kwargs["k"] = 3

# We define some variables that will be used inside our custom tool
CUSTOM_TOOL_DOCS_SEPARATOR ="\n---------------\n" # how to join together the
# retrieved docs to form a single string

# This is the function that defines our custom tool that retrieves relevant
# docs from Deep Lake
def retrieve_n_docs_tool(query: str) -> str:
    """Searches for relevant documents that may contain the answer to the query."""
    docs = retriever.get_relevant_documents(query)
    texts = [doc.page_content for doc in docs]
    texts_merged = "---------------\n" + CUSTOM_TOOL_DOCS_SEPARATOR.join(texts) + "\n---------------"
    return texts_merged

In [37]:
from langchain_core.tools import Tool


# We create the tool that uses the "retrieve_n_docs_tool" function
tools = [
    Tool(
        name="Search Private Docs",
        func=retrieve_n_docs_tool,
        description="Useful for when you need to answer questions about current events about Artificial Intelligence"
    )
]

In [48]:
from langchain_openai import ChatOpenAI
from langchain_experimental.plan_and_execute.agent_executor import PlanAndExecute
from langchain_experimental.plan_and_execute.executors.agent_executor import load_agent_executor
from langchain_experimental.plan_and_execute.planners.chat_planner import load_chat_planner
# let's create the Plan and Execute agent
# Initialize the LLM model
model = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Load planner and executor
planner = load_chat_planner(model)
executor = load_agent_executor(llm=model, tools=tools, verbose=True)



In [49]:
agent = PlanAndExecute(planner=planner, executor=executor, verbose=True)

In [50]:
# we test the agent
response = agent.run("Write an overview of Artificial Intelligence regulations by governments by country")

C:\Users\prave\AppData\Local\Temp\ipykernel_4640\2399054176.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  response = agent.run("Write an overview of Artificial Intelligence regulations by governments by country")




> Entering new PlanAndExecute chain...
steps=[Step(value='Research and gather information on Artificial Intelligence regulations by governments in different countries.'), Step(value='Organize the information by country, including details such as key regulations, policies, and guidelines related to Artificial Intelligence.'), Step(value='Summarize the information for each country in a concise manner.'), Step(value='Provide an overview of the regulations by comparing and contrasting the approaches taken by different countries.'), Step(value='Include any trends or common themes observed in the regulations across countries.'), Step(value='Check for the latest updates or changes in regulations to ensure the information is up-to-date.'), Step(value='Compile the overview into a comprehensive document or report.'), Step(value='Review the document for accuracy and clarity.'), Step(value="Make the overview accessible to the user.\nGiven the above steps taken, please respond to the user's origi

C:\Users\prave\AppData\Local\Temp\ipykernel_4640\2065998551.py:12: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents(query)



Observation: ---------------
“US-based AI developers will likely steal a march on their European competitors given the news that the EU parliamentary committees have green-lit its groundbreaking AI Act, where AI systems will need to be categorised according to their potential for harm from the outset. The US tech approach is typically to experiment first and, once market and product fit is established, to retrofit to other markets and their regulatory framework. This approach fosters innovation whereas EU-based AI developers will need to take note of the new rules and develop systems and processes which may take the edge off their ability to innovate. The UK is adopting a similar approach to the US, although the proximity of the EU market means that UK-based developers are more likely to fall into step with the EU ruleset from the outset. However, the potential to experiment in a safe space – a regulatory sandbox – may prove very attractive.”
---------------
To boost AI innovation, ME

In [52]:
print(response)

Please provide the original question for me to respond to.
